# Circuits Deep Dive

This notebook explains `src/cc_dqml/circuits.py` by inspecting the circuit constants, trainable parameter shapes, one forward pass through the PennyLane QNode, and the final probability-to-score interpretation step.

In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(path for path in candidates if (path / "src" / "cc_dqml").exists())
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".cache" / "matplotlib"))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

config_path = repo_root / "config" / "smoke.yaml"
if not config_path.exists():
    config_path = repo_root / "config-example" / "cc_dqml.yaml"

repo_root, config_path

(PosixPath('/Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC'),
 PosixPath('/Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC/config/smoke.yaml'))

## Imports

`circuits.py` is deliberately small: it builds an 8-qubit PennyLane circuit, initializes the trainable parameter arrays, and maps the 2-readout-qubit probability vector to a scalar score.

In [2]:
import numpy as np

from cc_dqml.config import load_config
from cc_dqml.data import generate_synthetic_dataset
from cc_dqml.circuits import (
    N_QUBITS,
    READOUT_WIRES,
    PARAMETER_NAMES,
    initialize_parameters,
    interpret_probabilities,
    make_cc_dqml_qnode,
    parameter_shapes,
    parameters_to_dict,
)

config = load_config(config_path)
config

ExperimentConfig(experiment=ExperimentSettings(model='cc_dqml', output_dir=PosixPath('results/smoke_cc_dqml'), dataset_seed=0, init_seed=0), data=DataSettings(n_features=8, n_samples=64, n_clusters=8, train_size=48, validation_size=16, sphere_radius=0.7853981633974483), model=ModelSettings(qpus=2, qubits_per_qpu=4, convolutional_sub_layers=1, communication='classical', interpret_weights_initial=(1.0, -1.0, -1.0, 1.0)), training=TrainingSettings(optimizer='adam', learning_rate=0.05, batch_size=8, iterations=2))

## Circuit constants

The prototype models two 4-qubit QPUs as one 8-qubit simulator. The readout uses one wire from each QPU, so `qml.probs(wires=[3, 7])` returns four probabilities.

In [3]:
constants = {
    "N_QUBITS": N_QUBITS,
    "READOUT_WIRES": READOUT_WIRES,
    "PARAMETER_NAMES": PARAMETER_NAMES,
}
constants

{'N_QUBITS': 8,
 'READOUT_WIRES': [3, 7],
 'PARAMETER_NAMES': ('conv', 'pool', 'feedforward', 'interpret')}

## Parameter shapes

`parameter_shapes` is the map from model depth to trainable array dimensions. The tuple returned by `initialize_parameters` stays in the same order as `PARAMETER_NAMES`.

In [4]:
shapes = parameter_shapes(config.model.convolutional_sub_layers)
params = initialize_parameters(
    config.model.convolutional_sub_layers,
    config.model.interpret_weights_initial,
    config.experiment.init_seed,
)
named_params = parameters_to_dict(params)

{name: {"expected": shapes[name], "actual": tuple(value.shape)} for name, value in named_params.items()}

{'conv': {'expected': (1, 2, 4), 'actual': (1, 2, 4)},
 'pool': {'expected': (2, 3, 4), 'actual': (2, 3, 4)},
 'feedforward': {'expected': (4, 4), 'actual': (4, 4)},
 'interpret': {'expected': (4,), 'actual': (4,)}}

In [5]:
for name, value in named_params.items():
    print(f"{name:12s} shape={tuple(value.shape)} requires_grad={getattr(value, 'requires_grad', None)}")

conv         shape=(1, 2, 4) requires_grad=True
pool         shape=(2, 3, 4) requires_grad=True
feedforward  shape=(4, 4) requires_grad=True
interpret    shape=(4,) requires_grad=True


## One forward pass

A single input sample has 8 features. `_embed_features` splits those features across the two local QPUs, applies local rotations and ring-style `IsingZZ` interactions, then the QNode returns the readout probabilities.

In [6]:
dataset = generate_synthetic_dataset(config.data, config.experiment.dataset_seed)
x_sample = dataset.x_train[0]
y_sample = dataset.y_train[0]

x_sample.shape, y_sample, np.round(x_sample, 3)

((8,),
 np.float64(-1.0),
 array([-0.988, -0.927, -1.083, -0.542, -0.464, -1.039, -1.024,  0.428]))

In [7]:
circuit = make_cc_dqml_qnode()
probabilities = circuit(
    x_sample,
    named_params["conv"],
    named_params["pool"],
    named_params["feedforward"],
)
score = interpret_probabilities(probabilities, named_params["interpret"])

probabilities, score

(tensor([0.25021842, 0.28820503, 0.22585127, 0.23572529], requires_grad=True),
 np.float64(-0.028112584347733144))

## Sanity checks

The two readout qubits produce four probabilities, those probabilities should sum to 1, and the interpretation layer should return one scalar score.

In [8]:
probabilities_np = np.asarray(probabilities, dtype=float)
score_np = np.asarray(score, dtype=float)

assert probabilities_np.shape == (4,)
assert np.isclose(probabilities_np.sum(), 1.0)
assert score_np.shape == ()

{
    "probability_shape": probabilities_np.shape,
    "probability_sum": float(probabilities_np.sum()),
    "score": float(score_np),
}

{'probability_shape': (4,),
 'probability_sum': 0.9999999999999964,
 'score': -0.028112584347733144}